# 13 — BiomedCLIP query embedding and FAISS retrieval timing

This notebook addresses Reviewer 2, Minor Comment 2. It measures image
loading/preprocessing, BiomedCLIP query encoding, raw IndexFlatIP cosine
search, and the leakage-safe retrieval wrapper separately. It reports
median, interquartile range, mean, and 95th percentile on the executing
hardware. The model identifier is taken from the original FAISS build
configuration. Model loading and warm-up are excluded from per-query
latency. By default, all 1,034 unique test images are measured three
times and FAISS uses one CPU thread.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [
    entry for entry in sys.path
    if Path(entry or ".").resolve() != implementation_dir
]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
POST_ROOT = PATHS["root"] / "post_rerun"
POST_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", RERUN_DIR)
print("Output:", POST_ROOT)

In [ ]:
import importlib.metadata, platform, random, time
from datetime import datetime, timezone
import numpy as np, pandas as pd
from PIL import Image
import torch
try:
    import open_clip
except ImportError as exc:
    raise ImportError(
        "Install open-clip-torch in the Biowulf environment, then restart the kernel."
    ) from exc
import faiss
from rerun_code.common import write_json
from rerun_code.config import sha256_path
from rerun_code.leakage_safe_retrieval import (
    load_gallery_bundle, load_query_bundle, safe_search,
)

OUTPUT = POST_ROOT / "retrieval_timing"
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get(
    "JAMIA_BIOMEDCLIP_MODEL_ID",
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
)
REPEATS = int(os.environ.get("JAMIA_TIMING_REPEATS", "3"))
MAX_QUERIES = int(os.environ.get("JAMIA_TIMING_MAX_QUERIES", "0"))
WARMUP = int(os.environ.get("JAMIA_TIMING_WARMUP", "10"))
FAISS_THREADS = int(os.environ.get("JAMIA_FAISS_THREADS", "1"))
SEED = int(CONFIG["statistics"]["seed"])
MIN_EMBEDDING_COSINE = float(os.environ.get("JAMIA_MIN_EMBEDDING_COSINE", "0.99"))
if REPEATS < 1 or FAISS_THREADS < 1:
    raise ValueError("JAMIA_TIMING_REPEATS and JAMIA_FAISS_THREADS must be positive")
if not torch.cuda.is_available() and os.environ.get("JAMIA_ALLOW_CPU_TIMING", "0") != "1":
    raise RuntimeError(
        "A CUDA GPU is required for manuscript timing. Set JAMIA_ALLOW_CPU_TIMING=1 only for a smoke test."
    )
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
faiss.omp_set_num_threads(FAISS_THREADS)
print("Device:", DEVICE, "FAISS threads:", FAISS_THREADS, "repeats:", REPEATS)

In [ ]:
# Load the exact BiomedCLIP encoder recorded by the original FAISS build.
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_ID)
model = model.to(DEVICE).eval()

combined_vectors, combined_metadata = load_query_bundle(PATHS["bundles"] / "combined" / "queries")
if len(combined_vectors) != 1034:
    print("Warning: expected 1,034 combined queries; observed", len(combined_vectors))
unique_queries = []
seen = set()
for vector, metadata in zip(combined_vectors, combined_metadata):
    record_id = str(metadata["record_id"])
    if record_id in seen:
        continue
    seen.add(record_id)
    unique_queries.append((record_id, vector, metadata))
if MAX_QUERIES > 0:
    rng = random.Random(SEED)
    rng.shuffle(unique_queries)
    unique_queries = unique_queries[:MAX_QUERIES]

def resolve_image(metadata):
    original = Path(str(metadata.get("image_path", "")))
    dataset = str(metadata["dataset"])
    candidates = [
        original,
        Path(CONFIG["datasets"][dataset]["test"]) / original.name,
        Path(CONFIG["datasets"][dataset]["test"]) / str(metadata.get("image_id", "")),
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Cannot resolve image for {metadata['record_id']}: {candidates}")

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

# Warm-up is not included in timing.
with torch.inference_mode():
    for _, _, metadata in unique_queries[:min(WARMUP, len(unique_queries))]:
        with Image.open(resolve_image(metadata)) as image:
            tensor = preprocess(image.convert("RGB")).unsqueeze(0).to(DEVICE)
        _ = model.encode_image(tensor)
    synchronize()
print("Warm-up complete:", min(WARMUP, len(unique_queries)), "images")

In [ ]:
embedding_rows = []
generated_vectors = {}
for repeat in range(REPEATS):
    order = list(unique_queries)
    random.Random(SEED + repeat).shuffle(order)
    with torch.inference_mode():
        for index, (record_id, stored_vector, metadata) in enumerate(order, start=1):
            start = time.perf_counter_ns()
            with Image.open(resolve_image(metadata)) as image:
                tensor = preprocess(image.convert("RGB")).unsqueeze(0)
            after_preprocess = time.perf_counter_ns()
            tensor = tensor.to(DEVICE, non_blocking=False)
            synchronize()
            before_encode = time.perf_counter_ns()
            encoded = model.encode_image(tensor).float()
            encoded = encoded / encoded.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            synchronize()
            after_encode = time.perf_counter_ns()
            vector = encoded[0].detach().cpu().numpy().astype(np.float32)
            cosine = float(np.dot(vector, np.asarray(stored_vector, dtype=np.float32)))
            generated_vectors[record_id] = vector
            embedding_rows.append({
                "query_record_id": record_id,
                "source_dataset": metadata["dataset"],
                "repeat": repeat + 1,
                "preprocess_ms": (after_preprocess - start) / 1e6,
                "host_to_device_and_sync_ms": (before_encode - after_preprocess) / 1e6,
                "embedding_ms": (after_encode - before_encode) / 1e6,
                "query_encoding_total_ms": (after_encode - start) / 1e6,
                "cosine_vs_frozen_query_embedding": cosine,
            })
            if index % 100 == 0:
                print(f"Embedding repeat {repeat + 1}/{REPEATS}: {index}/{len(order)}")
embedding = pd.DataFrame(embedding_rows)
minimum_observed = float(embedding["cosine_vs_frozen_query_embedding"].min())
if minimum_observed < MIN_EMBEDDING_COSINE:
    raise AssertionError(
        f"New BiomedCLIP vectors do not reproduce the frozen query vectors: "
        f"minimum cosine={minimum_observed:.6f} < {MIN_EMBEDDING_COSINE}. "
        "Do not report timing until the original preprocessing/checkpoint is restored."
    )
embedding_path = OUTPUT / "query_embedding_timing.csv"
embedding.to_csv(embedding_path, index=False)
print("Minimum embedding reproducibility cosine:", minimum_observed)

In [ ]:
search_rows = []
for bundle in ("mimic", "iuhn", "combined"):
    bundle_root = PATHS["bundles"] / bundle
    index, gallery_metadata = load_gallery_bundle(bundle_root / "gallery")
    _, query_metadata = load_query_bundle(bundle_root / "queries")
    if MAX_QUERIES > 0:
        selected_ids = set(generated_vectors)
        query_metadata = [row for row in query_metadata if row["record_id"] in selected_ids]
    for repeat in range(REPEATS):
        order = list(query_metadata)
        random.Random(SEED + 1000 + repeat).shuffle(order)
        for position, metadata in enumerate(order, start=1):
            record_id = str(metadata["record_id"])
            vector = generated_vectors[record_id].reshape(1, -1)
            start_raw = time.perf_counter_ns()
            raw_scores, raw_indices = index.search(vector, int(CONFIG["retrieval_k"]))
            end_raw = time.perf_counter_ns()
            start_safe = time.perf_counter_ns()
            neighbors = safe_search(
                index, gallery_metadata, vector[0], metadata,
                k=int(CONFIG["retrieval_k"]),
                phash_threshold=int(CONFIG["phash_threshold"]),
            )
            end_safe = time.perf_counter_ns()
            if len(neighbors) != int(CONFIG["retrieval_k"]):
                raise AssertionError("Leakage-safe retrieval returned the wrong number of neighbors")
            search_rows.append({
                "bundle": bundle,
                "source_dataset": metadata["dataset"],
                "query_record_id": record_id,
                "repeat": repeat + 1,
                "gallery_size": int(index.ntotal),
                "k": int(CONFIG["retrieval_k"]),
                "raw_faiss_indexflatip_ms": (end_raw - start_raw) / 1e6,
                "leakage_safe_search_ms": (end_safe - start_safe) / 1e6,
                "raw_rank1_cosine": float(raw_scores[0, 0]),
            })
            if position % 250 == 0:
                print(f"Search {bundle} repeat {repeat + 1}/{REPEATS}: {position}/{len(order)}")
search = pd.DataFrame(search_rows)
search_path = OUTPUT / "faiss_search_timing.csv"
search.to_csv(search_path, index=False)

In [ ]:
def summarize(frame, group_columns, value_columns, stage):
    rows = []
    for keys, group in frame.groupby(group_columns, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        fixed = dict(zip(group_columns, keys))
        for value_column in value_columns:
            values = pd.to_numeric(group[value_column], errors="coerce").dropna()
            q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
            rows.append({
                **fixed, "stage": stage, "measurement": value_column,
                "n_measurements": len(values), "median_ms": median,
                "q1_ms": q1, "q3_ms": q3, "iqr_ms": q3 - q1,
                "mean_ms": values.mean(), "p95_ms": values.quantile(0.95),
            })
    return rows

summary_rows = []
summary_rows.extend(summarize(
    embedding, ["source_dataset"],
    ["preprocess_ms", "embedding_ms", "query_encoding_total_ms"], "query_encoding",
))
summary_rows.extend(summarize(
    search, ["bundle", "source_dataset"],
    ["raw_faiss_indexflatip_ms", "leakage_safe_search_ms"], "search",
))
end_to_end = search.merge(
    embedding[["query_record_id", "repeat", "query_encoding_total_ms"]],
    on=["query_record_id", "repeat"], validate="many_to_one",
)
end_to_end["embedding_plus_safe_retrieval_ms"] = (
    end_to_end["query_encoding_total_ms"] + end_to_end["leakage_safe_search_ms"]
)
summary_rows.extend(summarize(
    end_to_end, ["bundle", "source_dataset"],
    ["embedding_plus_safe_retrieval_ms"], "end_to_end_retrieval",
))
summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT / "retrieval_timing_summary.csv"
summary.to_csv(summary_path, index=False)

complete_cohort = MAX_QUERIES <= 0
status = {
    "ready_for_manuscript": complete_cohort and DEVICE.type == "cuda",
    "run_scope": "full" if complete_cohort else f"smoke_test_{MAX_QUERIES}_queries",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_loading_and_warmup_excluded": True,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU",
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "open_clip_torch_version": importlib.metadata.version("open_clip_torch"),
    "faiss_version": getattr(faiss, "__version__", "unknown"),
    "faiss_threads": FAISS_THREADS,
    "repeats": REPEATS,
    "warmup_queries": min(WARMUP, len(unique_queries)),
    "n_unique_queries": len(unique_queries),
    "minimum_cosine_vs_frozen_query_embedding": float(
        embedding["cosine_vs_frozen_query_embedding"].min()
    ),
    "outputs": {
        str(path): sha256_path(path)
        for path in (embedding_path, search_path, summary_path)
    },
    "reporting_note": (
        "Report query encoding, raw FAISS search, and leakage-safe retrieval separately; "
        "identify hardware, FAISS thread count, repeats, and IQR."
    ),
}
write_json(OUTPUT / "notebook13_status.json", status)
display(summary)
print(json.dumps(status, indent=2))